# JS08 - Tugas Lab 2: HDBSCAN pada Dataset Nyata

Pada tugas ini kita melakukan *clustering* menggunakan **HDBSCAN** terhadap dataset nyata dari `sklearn.datasets`, yaitu **Iris dataset**. Selanjutnya hasilnya dilaporkan berupa jumlah *cluster*, jumlah *noise*, visualisasi (dengan reduksi dimensi PCA), serta analisis apakah hasil clustering sesuai dengan label asli dataset.

## Langkah 0 - Import Library dan Instalasi HDBSCAN

In [ ]:
!pip install -q hdbscan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import hdbscan

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

## Langkah 1 - Load dan Inspeksi Data

Iris dataset berisi 150 sampel bunga iris (50 per spesies) dengan 4 fitur numerik dan 3 label target.

Fitur-fitur (semua bersatuan sentimeter): `sepal length`, `sepal width`, `petal length`, `petal width`. Karena keempat fitur memiliki satuan yang sama, *scaling* tidak terlalu diperlukan; data digunakan apa adanya.

In [ ]:
data = load_iris()
X = data.data
y = data.target

print('Shape data:', X.shape)
print('Nama fitur:', data.feature_names)
print('Nama target:', list(data.target_names))

pd.DataFrame(X, columns=data.feature_names).describe().round(2)

## Langkah 2 - Model HDBSCAN

HDBSCAN adalah pengembangan DBSCAN berbasis hierarki yang otomatis menentukan *cluster* berdasarkan kepadatan data, sehingga **tidak memerlukan penentuan jumlah cluster** (`n_clusters`). Parameter yang dipakai adalah nilai bawaan (`min_cluster_size=5`, `min_samples=None`).

Model menghasilkan label setiap titik: label `-1` menandakan *noise* (titik yang tidak masuk cluster mana pun).

In [ ]:
hdb = hdbscan.HDBSCAN()
labels = hdb.fit_predict(X)

n_cluster = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print(f'Jumlah cluster yang terbentuk : {n_cluster}')
print(f'Jumlah noise                  : {n_noise} ({n_noise / len(labels) * 100:.2f}%)')

print('\nJumlah anggota tiap cluster (-1 = noise):')
print(pd.Series(labels).value_counts().sort_index())

## Langkah 3 - Evaluasi dan Perbandingan dengan Label Asli

Untuk menilai apakah hasil clustering sesuai dengan label asli, kita menghitung *silhouette score*, *Davies-Bouldin Index* (DBI), serta *Adjusted Rand Index* (ARI) yang mengukur kesesuaian antara label hasil clustering dan label asli (nilai 1 = sempurna, 0 = acak).

In [ ]:
print(f'Silhouette Score        : {silhouette_score(X, labels):.4f}')
print(f'Davies-Bouldin Index    : {davies_bouldin_score(X, labels):.4f}')
print(f'Adjusted Rand Index     : {adjusted_rand_score(y, labels):.4f}')

In [ ]:
ct = pd.crosstab(labels, y, rownames=['Cluster HDBSCAN'], colnames=['Label Asli (Spesies)'])
ct.columns = list(data.target_names)
ct

## Langkah 4 - Visualisasi dengan PCA

Iris berdimensi 4 fitur, sedangkan visualisasi hanya nyaman dilakukan hingga 2-3 dimensi. Karena itu kita reduksi dimensi menggunakan **PCA** (2 komponen utama) untuk mempermudah pengamatan bentuk *cluster*.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

print('Rasio variansi yang dijelaskan kedua komponen:')
print(pca.explained_variance_ratio_.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel kiri: hasil clustering HDBSCAN
ax = axes[0]
unique = np.unique(labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique))]
for k, col in zip(unique, colors):
    mask = labels == k
    if k == -1:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], marker='x', color='black',
                   s=60, label='Noise')
    else:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=40, alpha=0.8,
                   color=col, label=f'Cluster {int(k)}')
ax.set_title('Hasil Clustering HDBSCAN')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()

# Panel kanan: label asli
ax = axes[1]
for i, sp in enumerate(data.target_names):
    mask = y == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=40, alpha=0.8, label=sp)
ax.set_title('Label Asli (Spesies)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()

plt.tight_layout()
plt.show()

## Analisis: Apakah Hasil HDBSCAN Sesuai dengan Label Asli?

**Hasil HDBSCAN tidak sepenuhnya sesuai dengan label asli, tetapi sebagian.** HDBSCAN membentuk **2 cluster tanpa noise**, sedangkan label asli memiliki **3 spesies**.

Berdasarkan *crosstab*:

- **Cluster 0** beranggotakan **50 sampel setosa** secara tepat — sesuai sempurna dengan label asli `setosa`.
- **Cluster 1** merupakan **gabungan 50 versicolor + 50 virginica** — dua spesies ini tidak terpisah oleh HDBSCAN.

Hal ini terjadi karena **setosa terpisah jelas** dari dua spesies lainnya dalam ruang fitur, sedangkan **versicolor dan virginica saling tumpang-tindih** (kepadatannya saling menyambung). Karena HDBSCAN bekerja berbasis kepadatan, ia hanya mampu memisahkan kelompok yang padat dan terpisah, sehingga menggabungkan versicolor dan virginica menjadi satu *cluster*.

Poin evaluasi:

- **ARI 0.5681** (skala 0-1) menunjukkan kesesuaian sedang — memang setosa teridentifikasi sempurna, tetapi dua spesies lain digabung.
- **Silhouette Score 0.6867** dan **DBI 0.3828** tetap terlihat baik karena menggabungkan dua spesies yang tumpang-tindih justru menghasilkan satu *cluster* yang kompak, sehingga metrik internal kepadatan terlihat bagus.

Kesimpulannya, HDBSCAN berhasil menemukan struktur yang benar-benar terpisah (setosa) dan menandai bahwa versicolor-virginica merupakan satu kawasan yang menyambung. Dengan demikian, secara karakteristik *density-based clustering*, hasil HDBSCAN **sebagian sesuai** dengan label asli, dan ketidaksesuaiannya justru merefleksikan adanya *overlap* nyata pada data Iris.